[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ame5003-nlp/blob/main/primers/primer-4-numpy.ipynb)

**Click the badge above to open this lab in Google Colab.** Then choose *File → Save a copy in Drive* so your work is saved.

# Primer 4 — NumPy, Hands On

**MSIS · AME 5003 and AME 5053 · Practice notebook · about 1 hour · not assessed**

*A worked pass through the freeCodeCamp NumPy tutorial: what an array is, how it is indexed and
sliced, how it is built, and the arithmetic, statistics, reshaping and masking that come with it.*

This notebook follows one video and nothing else:

> **Python NumPy Tutorial for Beginners** — freeCodeCamp.org, tutorial by Keith Galli.
> <https://www.youtube.com/watch?v=QUT1VHiLmmI> · code at <https://github.com/KeithGalli/NumPy>

The video is about an hour long and works through a live notebook. This notebook covers the same
ground in the same order, with the output of every cell recorded next to it so the notebook still
reads on a phone with nothing running. Its sections are the video's own chapters:

| video time | section here |
| --- | --- |
| 01:15 | Part 1 — What NumPy is |
| 01:35 | Part 2 — NumPy against lists: speed and functionality |
| 09:17 | Part 3 — Where NumPy is used |
| 11:08 | Part 4 — The basics: creating arrays, shape, size, data type |
| 16:08 | Part 5 — Accessing and changing elements, rows and columns |
| 23:14 | Part 6 — Initialising different arrays |
| 31:34 | Part 7 — Problem 1 |
| 33:42 | Part 8 — Be careful when copying |
| 35:45 | Part 9 — Basic mathematics |
| 38:20 | Part 10 — Linear algebra |
| 42:19 | Part 11 — Statistics |
| 43:57 | Part 12 — Reorganising arrays |
| 47:29 | Part 13 — Loading data from a file |
| 50:20 | Part 14 — Advanced indexing and boolean masking |
| 55:59 | Part 15 — Problem 2 |

Run it in Colab or locally. NumPy is the only requirement, and it ships with both.


---
## Part 1 — What NumPy is

NumPy is a library for numerical computing in Python, and the thing it provides is the
**multidimensional array**: a grid of numbers, all of one type, held in one block of memory.
Almost everything else in the library is an operation on that grid.

The import line is a convention, but a universal one. Every piece of NumPy code you will read
says `np`.


In [ ]:
import numpy as np

print(np.__version__)
# Verified output:
#   2.5.1

---
## Part 2 — NumPy against lists

Python already has a container of numbers, so the first question is why a second one exists. The
video gives two answers: NumPy is faster, and it can do more.

### Why it is faster

Three reasons, and they compound.

**Fixed types.** A Python integer is a full object. Alongside its value it carries a reference
count, an object type and a size, so a single small integer costs on the order of 28 bytes. A
NumPy array stores the value alone, in a type you can choose — 4 bytes for `int32`, or 1 byte for
`int8` if the numbers are small. Less memory to read means less time reading it.


In [ ]:
import sys

# A Python int is an object, with bookkeeping attached to every element.
print("one Python int:  ", sys.getsizeof(1), "bytes")

# A NumPy array stores bare values, and you choose how wide they are.
a32 = np.array([1, 2, 3], dtype="int32")
a8 = np.array([1, 2, 3], dtype="int8")
print("int32 element:   ", a32.itemsize, "bytes")
print("int8 element:    ", a8.itemsize, "bytes")
print("whole int32 array:", a32.nbytes, "bytes")
# Verified output:
#   one Python int:   28 bytes
#   int32 element:    4 bytes
#   int8 element:     1 bytes
#   whole int32 array: 12 bytes

**No type checking.** Because every element of an array has the same type, NumPy does not have to
ask what each element is before working with it. A Python list can hold anything, so it must ask,
every time, for every element.

**Contiguous memory.** A list holds pointers scattered across memory; an array is one unbroken
block. That lets the CPU use vector instructions that act on several values at once, and it keeps
the values that are used together in the same cache line.

Put together, the loop that is written in Python disappears into compiled code.


In [ ]:
import time

n = 1_000_000
py_list = list(range(n))
np_array = np.arange(n)

start = time.perf_counter()
doubled = [x * 2 for x in py_list]        # a Python-level loop, one object at a time
list_time = time.perf_counter() - start

start = time.perf_counter()
doubled = np_array * 2                    # one call; the loop runs in compiled code
array_time = time.perf_counter() - start

print(f"list comprehension {list_time * 1000:7.1f} ms")
print(f"numpy              {array_time * 1000:7.1f} ms")
print(f"ratio              {list_time / array_time:7.0f}x")
# Your numbers will differ from run to run and machine to machine; the ratio is the point.

### Why it can do more

Multiplying two lists element by element is not something a list does — `[1,3,5] * [1,2,3]` is a
`TypeError`. Multiplying two arrays element by element is what `*` means. That difference runs
through the whole library: most of what you would write a loop for, NumPy already has as an
operation on whole arrays.


In [ ]:
left = np.array([1, 3, 5])
right = np.array([1, 2, 3])

print("element by element:", left * right)

# The list version has to say the same thing the long way round.
print("with lists:        ", [x * y for x, y in zip([1, 3, 5], [1, 2, 3])])
# Verified output:
#   element by element: [ 1  6 15]
#   with lists:         [1, 6, 15]

---
## Part 3 — Where NumPy is used

The video lists four places, and they are worth knowing because they explain the shape of the
library.

- **Mathematics.** Arrays, matrices and linear algebra, which is what people reach for when they
  would otherwise use MATLAB.
- **Plotting.** Matplotlib takes arrays, so anything you compute here is one call from a figure.
- **As a backend.** Pandas is built on NumPy, and so is a great deal of scientific Python.
  Storing a game board or an image as an array is the same idea: an image is a grid of pixel
  values, and cropping it is a slice.
- **Machine learning.** Every tensor library borrows the array's vocabulary — shape, axis,
  broadcasting — so learning it once pays twice.

---
## Part 4 — The basics

An array is made from a list, and it reports on itself. These five attributes are the ones to
know by name.


In [ ]:
a = np.array([1, 2, 3], dtype="int32")     # dtype is optional; here we ask for 32-bit ints
print(a)

b = np.array([[9.0, 8.0, 7.0],
              [6.0, 5.0, 4.0]])            # a 2-D array: a list of rows
print(b)
# Verified output:
#   [1 2 3]
#   [[9. 8. 7.]
#    [6. 5. 4.]]

In [ ]:
print("a.ndim:    ", a.ndim)       # number of dimensions
print("b.ndim:    ", b.ndim)
print("b.shape:   ", b.shape)      # one entry per dimension: (rows, columns)
print("a.dtype:   ", a.dtype)      # the one type shared by every element
print("b.dtype:   ", b.dtype)      # floats, because we typed 9.0 rather than 9
print("a.itemsize:", a.itemsize)   # bytes per element
print("a.size:    ", a.size)       # how many elements
print("a.nbytes:  ", a.nbytes)     # total bytes = size * itemsize
# Verified output:
#   a.ndim:     1
#   b.ndim:     2
#   b.shape:    (2, 3)
#   a.dtype:    int32
#   b.dtype:    float64
#   a.itemsize: 4
#   a.size:     3
#   a.nbytes:   12

Read `(2, 3)` as "two rows of three". Rows come first in every shape tuple you will meet.

`dtype` is worth a second look whenever a result surprises you. An array of integers divides
differently from an array of floats, and asking for `int8` when the data will not fit in a byte
is a mistake NumPy will let you make.

---
## Part 5 — Accessing and changing elements, rows and columns

Indexing takes one index per dimension, rows first, and every index counts from zero. Negative
indices count from the end.


In [ ]:
a = np.array([[1, 2, 3, 4, 5, 6, 7],
              [8, 9, 10, 11, 12, 13, 14]])
print(a)
print("shape:", a.shape)
# Verified output:
#   [[ 1  2  3  4  5  6  7]
#    [ 8  9 10 11 12 13 14]]
#   shape: (2, 7)

In [ ]:
print("a[1, 5]  ->", a[1, 5])      # row 1, column 5
print("a[1, -2] ->", a[1, -2])     # the same element, counting from the right
# Verified output:
#   a[1, 5]  -> 13
#   a[1, -2] -> 13

A colon means "every index along this dimension", which is how a whole row or a whole column is
taken.


In [ ]:
print("row 0:   ", a[0, :])        # every column of row 0
print("column 2:", a[:, 2])        # every row of column 2
# Verified output:
#   row 0:    [1 2 3 4 5 6 7]
#   column 2: [ 3 10]

A slice is `start:stop:step`, with `stop` excluded, exactly as it is for a Python list. The video's
example takes row 0 from index 1 up to but not including the last element, every second one.


In [ ]:
print(a[0, 1:-1:2])                # start 1, stop -1 (excluded), step 2
# Verified output:
#   [2 4 6]

Assignment works everywhere indexing does. Assigning a single number to a whole row or column
sets every element of it; assigning a list of the right length sets them one by one.


In [ ]:
a[1, 5] = 20                       # one element
a[:, 2] = 5                        # a whole column, all to the same value
print(a)

a[:, 2] = [1, 2]                   # a whole column, one value per row
print(a)
# Verified output:
#   [[ 1  2  5  4  5  6  7]
#    [ 8  9  5 11 12 20 14]]
#   [[ 1  2  1  4  5  6  7]
#    [ 8  9  2 11 12 20 14]]

### Three dimensions

The same rules, one more index. Work from the outside in: the first index chooses which
two-dimensional block, the second chooses the row inside it, the third the column.


In [ ]:
b = np.array([[[1, 2], [3, 4]],
              [[5, 6], [7, 8]]])
print(b)
print("shape:", b.shape)           # 2 blocks, each 2 rows of 2
# Verified output:
#   [[[1 2]
#     [3 4]]
#
#    [[5 6]
#     [7 8]]]
#   shape: (2, 2, 2)

In [ ]:
print("b[0, 1, 1] ->", b[0, 1, 1])  # first block, second row, second column
# Verified output:
#   b[0, 1, 1] -> 4

Replacing a slice of a 3-D array needs a replacement of the same shape. `b[:, 1, :]` picks the
second row of both blocks, which is a 2×2 region, so it takes a 2×2 replacement — this is the
point in the video where a mismatched list raises an error.


In [ ]:
b[:, 1, :] = [[9, 9], [8, 8]]      # 2 blocks x 2 columns, so two pairs
print(b)
# Verified output:
#   [[[1 2]
#     [9 9]]
#
#    [[5 6]
#     [8 8]]]

---
## Part 6 — Initialising different arrays

You rarely type an array out. These are the constructors the video uses.


In [ ]:
print(np.zeros(5))                 # 1-D of zeros
print(np.zeros((2, 3)))            # a shape tuple gives more dimensions
# Verified output:
#   [0. 0. 0. 0. 0.]
#   [[0. 0. 0.]
#    [0. 0. 0.]]

In [ ]:
print(np.ones((4, 2, 2), dtype="int32"))   # 4 blocks of 2x2, integers rather than floats
# Verified output:
#   [[[1 1]
#     [1 1]]
#
#    [[1 1]
#     [1 1]]
#
#    [[1 1]
#     [1 1]]
#
#    [[1 1]
#     [1 1]]]

In [ ]:
print(np.full((2, 2), 99))         # any other fill value

# full_like borrows the shape (and dtype) of an array you already have.
template = np.array([[1, 2, 3, 4, 5, 6, 7],
                     [8, 9, 10, 11, 12, 13, 14]])
print(np.full_like(template, 4))
# Verified output:
#   [[99 99]
#    [99 99]]
#   [[4 4 4 4 4 4 4]
#    [4 4 4 4 4 4 4]]

In [ ]:
np.random.seed(0)                  # seeded so this notebook prints the same numbers every run

print(np.random.rand(4, 2))        # decimals in [0, 1), one argument per dimension
print(np.random.randint(-4, 8, size=(3, 3)))   # integers from -4 up to but excluding 8
# Verified output:
#   [[0.5488135  0.71518937]
#    [0.60276338 0.54488318]
#    [0.4236548  0.64589411]
#    [0.43758721 0.891773  ]]
#   [[ 4  6 -3]
#    [ 2  3  3]
#    [ 4 -3  1]]

In [ ]:
print(np.identity(5))              # the identity matrix: square, ones on the diagonal
# Verified output:
#   [[1. 0. 0. 0. 0.]
#    [0. 1. 0. 0. 0.]
#    [0. 0. 1. 0. 0.]
#    [0. 0. 0. 1. 0.]
#    [0. 0. 0. 0. 1.]]

`np.repeat` repeats an array along an axis. Note the double brackets when building `arr`: it is a
1×3 array rather than a flat one, so `axis=0` has rows to repeat.


In [ ]:
arr = np.array([[1, 2, 3]])        # shape (1, 3), not (3,)
r1 = np.repeat(arr, 3, axis=0)     # repeat down the rows
print(r1)
# Verified output:
#   [[1 2 3]
#    [1 2 3]
#    [1 2 3]]

---
## Part 7 — Problem 1

Build this array:

```
1 1 1 1 1
1 0 0 0 1
1 0 9 0 1
1 0 0 0 1
1 1 1 1 1
```

Try it before reading on. The video's solution builds the pieces separately and drops one into
the other with a slice.


In [ ]:
output = np.ones((5, 5), dtype="int32")    # the border, and everything else for now
print(output)

z = np.zeros((3, 3), dtype="int32")        # the inside
z[1, 1] = 9                                # the middle of the inside
print(z)

output[1:-1, 1:-1] = z                     # rows 1..3 and columns 1..3, the inner square
print(output)
# Verified output:
#   [[1 1 1 1 1]
#    [1 1 1 1 1]
#    [1 1 1 1 1]
#    [1 1 1 1 1]
#    [1 1 1 1 1]]
#   [[0 0 0]
#    [0 9 0]
#    [0 0 0]]
#   [[1 1 1 1 1]
#    [1 0 0 0 1]
#    [1 0 9 0 1]
#    [1 0 0 0 1]
#    [1 1 1 1 1]]

---
## Part 8 — Be careful when copying

`b = a` does not make a second array. It makes a second name for the same one, so writing through
either name changes what both of them see. This is the same rule Python applies to lists, and it
is still the mistake that costs the most time.


In [ ]:
a = np.array([1, 2, 3])
b = a                              # a second name, not a second array
b[0] = 100

print("a:", a)                     # a changed too
print("b:", b)
# Verified output:
#   a: [100   2   3]
#   b: [100   2   3]

In [ ]:
a = np.array([1, 2, 3])
b = a.copy()                       # an independent array with the same contents
b[0] = 100

print("a:", a)                     # a is untouched
print("b:", b)
# Verified output:
#   a: [1 2 3]
#   b: [100   2   3]

---
## Part 9 — Basic mathematics

Arithmetic between an array and a number applies to every element, and produces a new array. The
original is not modified unless you assign back to it.


In [ ]:
a = np.array([1, 2, 3, 4])

print("a + 2 ->", a + 2)
print("a - 2 ->", a - 2)
print("a * 2 ->", a * 2)
print("a / 2 ->", a / 2)           # division always gives floats
print("a ** 2->", a ** 2)
print("a     ->", a)               # unchanged: none of the above assigned to it
# Verified output:
#   a + 2 -> [3 4 5 6]
#   a - 2 -> [-1  0  1  2]
#   a * 2 -> [2 4 6 8]
#   a / 2 -> [0.5 1.  1.5 2. ]
#   a ** 2-> [ 1  4  9 16]
#   a     -> [1 2 3 4]

Arithmetic between two arrays of the same shape pairs the elements up.


In [ ]:
b = np.array([1, 0, 1, 0])

print("a + b ->", a + b)
print("a * b ->", a * b)
# Verified output:
#   a + b -> [2 2 4 4]
#   a * b -> [1 0 3 0]

In [ ]:
print("cos:", np.cos(a))           # trigonometry, elementwise
print("sin:", np.sin(a))
print("sqrt:", np.sqrt(a))
# Verified output:
#   cos: [ 0.54030231 -0.41614684 -0.9899925  -0.65364362]
#   sin: [ 0.84147098  0.90929743  0.14112001 -0.7568025 ]
#   sqrt: [1.         1.41421356 1.73205081 2.        ]

The full list of these is in the NumPy reference under mathematical routines. When you find
yourself writing a loop over an array, look there first — the operation usually already exists.

---
## Part 10 — Linear algebra

Elementwise multiplication is `*`. Matrix multiplication is `np.matmul`, and it has the usual
requirement: the number of columns on the left must equal the number of rows on the right, and
the result has the rows of the left and the columns of the right.


In [ ]:
a = np.ones((2, 3))
print(a)

b = np.full((3, 2), 2)
print(b)

# (2, 3) x (3, 2) -> (2, 2). The inner 3s match, and they disappear.
print(np.matmul(a, b))
# Verified output:
#   [[1. 1. 1.]
#    [1. 1. 1.]]
#   [[2 2]
#    [2 2]
#    [2 2]]
#   [[6. 6.]
#    [6. 6.]]

`np.linalg` holds the rest of linear algebra: determinant, trace, inverse, eigenvalues, singular
value decomposition, matrix norms. The determinant of the identity matrix is 1, which is the
video's check that the call does what it says.


In [ ]:
c = np.identity(3)
print(np.linalg.det(c))            # 1.0, as it must be for the identity
# Verified output:
#   1.0

---
## Part 11 — Statistics

`min`, `max`, `sum` and the rest work over the whole array by default. Passing `axis=` runs them
along one dimension instead, and the axis you name is the one that disappears from the shape.


In [ ]:
stats = np.array([[1, 2, 3],
                  [4, 5, 6]])
print(stats)
print("shape:", stats.shape)
# Verified output:
#   [[1 2 3]
#    [4 5 6]]
#   shape: (2, 3)

In [ ]:
print("min of everything: ", np.min(stats))
print("max of everything: ", np.max(stats))
print("sum of everything: ", np.sum(stats))
# Verified output:
#   min of everything:  1
#   max of everything:  6
#   sum of everything:  21

In [ ]:
# axis=1 collapses the columns: one value per row. (2, 3) -> (2,)
print("max along axis 1:", np.max(stats, axis=1))

# axis=0 collapses the rows: one value per column. (2, 3) -> (3,)
print("sum along axis 0:", np.sum(stats, axis=0))
print("min along axis 0:", np.min(stats, axis=0))
# Verified output:
#   max along axis 1: [3 6]
#   sum along axis 0: [5 7 9]
#   min along axis 0: [1 2 3]

If you cannot remember which axis is which, do not reason about rows and columns — print the
shape of the result. The axis you named is gone from it.

---
## Part 12 — Reorganising arrays

`reshape` gives the same data a different shape. Any shape works as long as it holds exactly as
many elements as the array already has.


In [ ]:
before = np.array([[1, 2, 3, 4],
                   [5, 6, 7, 8]])
print(before, before.shape)        # 8 elements

print(before.reshape((8, 1)))      # 8 rows of 1
print(before.reshape((4, 2)))      # 4 rows of 2
print(before.reshape((2, 2, 2)))   # 2 blocks of 2x2
# Verified output:
#   [[1 2 3 4]
#    [5 6 7 8]] (2, 4)
#   [[1]
#    [2]
#    [3]
#    [4]
#    [5]
#    [6]
#    [7]
#    [8]]
#   [[1 2]
#    [3 4]
#    [5 6]
#    [7 8]]
#   [[[1 2]
#     [3 4]]
#
#    [[5 6]
#     [7 8]]]

Asking for a shape that does not hold 8 elements is an error rather than a guess:

```python
before.reshape((2, 3))
```
```
ValueError: cannot reshape array of size 8 into shape (2,3)
```

Stacking joins arrays instead of reshaping one. `vstack` puts them one above another and `hstack`
side by side; the dimension being joined along has to line up.


In [ ]:
v1 = np.array([1, 2, 3, 4])
v2 = np.array([5, 6, 7, 8])

print(np.vstack([v1, v2]))         # two rows
print(np.vstack([v1, v2, v1, v2])) # four rows, and an array may be reused
# Verified output:
#   [[1 2 3 4]
#    [5 6 7 8]]
#   [[1 2 3 4]
#    [5 6 7 8]
#    [1 2 3 4]
#    [5 6 7 8]]

In [ ]:
h1 = np.ones((2, 4))
h2 = np.zeros((2, 2))

# Both have 2 rows, so they can sit side by side: (2, 4) + (2, 2) -> (2, 6).
print(np.hstack((h1, h2)))
# Verified output:
#   [[1. 1. 1. 1. 0. 0.]
#    [1. 1. 1. 1. 0. 0.]]

---
## Part 13 — Loading data from a file

`np.genfromtxt` reads a text file of numbers into an array, given the separator. The video reads
a `data.txt` of comma-separated integers; the cell below writes that same file first so this
notebook works anywhere, including a fresh Colab session.


In [ ]:
# The data file from the video, written here so nothing needs downloading.
with open("data.txt", "w") as f:
    f.write("1,13,21,11,196,75,4,3,34,6,7,8,0,1,2,3,4,5\n")
    f.write("3,42,12,33,766,75,4,55,6,4,3,4,5,6,7,0,11,12\n")
    f.write("1,22,33,11,999,11,2,1,78,0,1,2,9,8,7,1,76,88\n")

filedata = np.genfromtxt("data.txt", delimiter=",")
print(filedata.dtype)              # float64: genfromtxt reads floats by default
print(filedata)
# Verified output:
#   float64
#   [[  1.  13.  21.  11. 196.  75.   4.   3.  34.   6.   7.   8.   0.   1.
#       2.   3.   4.   5.]
#    [  3.  42.  12.  33. 766.  75.   4.  55.   6.   4.   3.   4.   5.   6.
#       7.   0.  11.  12.]
#    [  1.  22.  33.  11. 999.  11.   2.   1.  78.   0.   1.   2.   9.   8.
#       7.   1.  76.  88.]]

In [ ]:
filedata = filedata.astype("int32")    # astype returns a converted copy
print(filedata.dtype)
print(filedata)
# Verified output:
#   int32
#   [[  1  13  21  11 196  75   4   3  34   6   7   8   0   1   2   3   4   5]
#    [  3  42  12  33 766  75   4  55   6   4   3   4   5   6   7   0  11  12]
#    [  1  22  33  11 999  11   2   1  78   0   1   2   9   8   7   1  76  88]]

---
## Part 14 — Advanced indexing and boolean masking

Comparing an array with a number gives an array of the same shape full of `True` and `False`.
Nothing is filtered yet; this is only the test, applied to every element.


In [ ]:
print(filedata > 50)
# Verified output:
#   [[False False False False  True  True False False False False False False
#     False False False False False False]
#    [False False False False  True  True False  True False False False False
#     False False False False False False]
#    [False False False False  True False False False  True False False False
#     False False False False  True  True]]

Putting that boolean array inside the brackets is what filters. The result is one dimensional —
it is the elements that passed, in order, and their positions in the grid are not kept.


In [ ]:
print(filedata[filedata > 50])
# Verified output:
#   [196  75 766  75  55 999  78  76  88]

`np.any` and `np.all` reduce a boolean array to a single answer, or, with `axis=`, to one answer
per row or column.


In [ ]:
# One answer per column: is there any value over 50 in this column?
print(np.any(filedata > 50, axis=0))

# One answer per column: is every value in it over 50?
print(np.all(filedata > 50, axis=0))
# Verified output:
#   [False False False False  True  True False  True  True False False False
#    False False False False  True  True]
#   [False False False False  True False False False False False False False
#    False False False False False False]

Conditions combine with `&` for "and" and `|` for "or", and `~` negates. The brackets around each
comparison are not optional: `&` binds tighter than `>`, so without them the expression is
parsed the wrong way round.


In [ ]:
print((filedata > 50) & (filedata < 100))      # over 50 and under 100
print(~((filedata > 50) & (filedata < 100)))   # everything else
# Verified output:
#   [[False False False False False  True False False False False False False
#     False False False False False False]
#    [False False False False False  True False  True False False False False
#     False False False False False False]
#    [False False False False False False False False  True False False False
#     False False False False  True  True]]
#   [[ True  True  True  True  True False  True  True  True  True  True  True
#      True  True  True  True  True  True]
#    [ True  True  True  True  True False  True False  True  True  True  True
#      True  True  True  True  True  True]
#    [ True  True  True  True  True  True  True  True False  True  True  True
#      True  True  True  True False False]]

A list inside the brackets indexes with several positions at once, in the order you give them.


In [ ]:
a = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9])
print(a[[1, 2, 8]])                # elements at index 1, 2 and 8
# Verified output:
#   [2 3 9]

---
## Part 15 — Problem 2

Here is the array the video ends on: the numbers 1 to 30, in six rows of five.


In [ ]:
grid = np.arange(1, 31).reshape((6, 5))
print(grid)
# Verified output:
#   [[ 1  2  3  4  5]
#    [ 6  7  8  9 10]
#    [11 12 13 14 15]
#    [16 17 18 19 20]
#    [21 22 23 24 25]
#    [26 27 28 29 30]]

Index each of these three sets, without typing the numbers out:

1. `11 12` above `16 17`
2. `2 8 14 20` — the diagonal running down and to the right from row 0
3. `4 5` above `24 25` above `29 30`

Try them before running the cell.


In [ ]:
# 1. A rectangle: rows 2 to 3, columns 0 to 1. Both ends are slices, so the shape survives.
print(grid[2:4, 0:2])

# 2. A diagonal, so pair the indices up: (0,1), (1,2), (2,3), (3,4).
#    Two lists of the same length index element by element, and the result is 1-D.
print(grid[[0, 1, 2, 3], [1, 2, 3, 4]])

# 3. Rows 0, 4 and 5, and from column 3 to the end of each.
#    A list of rows with a slice of columns keeps the rectangle shape.
print(grid[[0, 4, 5], 3:])
# Verified output:
#   [[11 12]
#    [16 17]]
#   [ 2  8 14 20]
#   [[ 4  5]
#    [24 25]
#    [29 30]]

---
## What to remember

| you want | the call |
| --- | --- |
| what am I holding | `a.shape`, `a.dtype`, `a.ndim`, `a.size` |
| one element | `a[1, 5]`, and `a[0, 1, 1]` in three dimensions |
| a row, a column | `a[0, :]`, `a[:, 2]` |
| part of a row | `a[0, 1:-1:2]` — `start:stop:step`, `stop` excluded |
| a new array | `np.zeros`, `np.ones`, `np.full`, `np.full_like`, `np.identity`, `np.arange` |
| random numbers | `np.random.rand`, `np.random.randint` |
| an independent array | `a.copy()` — plain assignment only adds a name |
| elementwise arithmetic | `a + 2`, `a * b`, `a ** 2`, `np.cos(a)` |
| a matrix product | `np.matmul(a, b)` |
| a determinant, an inverse | `np.linalg.det`, `np.linalg.inv` |
| statistics | `np.min`, `np.max`, `np.sum`, with `axis=` to work along one dimension |
| a different shape | `a.reshape((4, 2))` |
| joining arrays | `np.vstack`, `np.hstack` |
| numbers from a file | `np.genfromtxt("data.txt", delimiter=",")`, then `.astype("int32")` |
| filtering | `a[a > 50]`, with `&`, `|`, `~` and brackets around each comparison |
| several positions at once | `a[[1, 2, 8]]` |

Three things from the video that are worth carrying away on their own. Assignment does not copy,
so use `.copy()` when you mean a second array. The axis you name in a statistic is the one that
disappears, so read the shape of the result rather than reasoning it out. And a boolean mask is
just an array of `True` and `False` — you can print it, which makes a filter that returns the
wrong rows something you can look at rather than guess about.

The video's own next step is the NumPy reference documentation, which is organised by the same
headings this notebook used: <https://numpy.org/doc/stable/reference/>.
